# ДЗ 2.2 — Перекос данных

Соединил просмотры с каталогом по element_uid. Проверил частоты ключей, NULL и повторы в каталоге.

## Варианты

Исходный JOIN запустил через SortMergeJoin без AQE, чтобы увидеть перекос без автоматической коррекции. Сравнил его с broadcast каталога и Salt Join.

Соль добавил ключам с долей от 1%: каждому просмотру назначил одно из 16 значений, соответствующие строки каталога размножил на все значения. Для остальных ключей оставил соль 0.

Сделал прогрев и по три замера полной записи в S3. Результаты сравнил через exceptAll в обе стороны, с учётом повторов. Код — task2.py.

In [1]:
from task2 import main
main()

task2:baseline:-1 63.508s


task2:broadcast:-1 33.595s


task2:salt:-1 42.314s


task2:baseline:0 63.248s


task2:broadcast:0 39.178s


task2:salt:0 62.361s


... (2 outputs left out)

task2:baseline:1 67.601s


task2:salt:2 42.321s


task2:baseline:2 70.774s


task2:broadcast:2 35.083s


In [2]:
from view_results import show_checks, show_measurements, show_plans
show_checks(2)

Строк после JOIN: 30,070,692
Broadcast и Salt Join совпали с исходным результатом по всем колонкам и повторам.


In [3]:
show_measurements(2)
show_plans(2)

baseline: 3 замеров, медиана 67.60 с (63.25–70.77), файлов 16.
  Shuffle read/write: 360.3/360.3 МиБ; spill memory/disk: 656.0/200.8 МиБ.
broadcast: 3 замеров, медиана 39.18 с (35.08–40.02), файлов 4.
  Shuffle read/write: 0.0/0.0 МиБ; spill memory/disk: 0.0/0.0 МиБ.
salt: 3 замеров, медиана 49.16 с (42.32–62.36), файлов 16.
  Shuffle read/write: 388.7/388.7 МиБ; spill memory/disk: 640.0/159.5 МиБ.
Все замеры: results/task2/measurements.csv
baseline: SortMergeJoin; AQE = False
broadcast: BroadcastHashJoin; AQE = False
salt: SortMergeJoin; AQE = False
Полные планы каждого замера: results/task2/plans/


## Вывод

В просмотрах 30 070 692 строки, в каталоге 10 200 уникальных ключей. NULL в ключах и просмотров без соответствия не нашёл. На один фильм, `element_uid = 2714`, приходится 68.16% просмотров.

После прогрева сделал по три замера:

- Исходный JOIN: медиана 67.60 с, разброс 63.25–70.77 с.
- Broadcast: медиана 39.18 с, разброс 35.08–40.02 с.
- Salt Join: медиана 49.16 с, разброс 42.32–62.36 с.

Выбрал бы broadcast: каталог небольшой — около 2.81 МиБ в JSON. На этом стенде он помещается в память и убирает shuffle просмотров. По медиане получилось быстрее исходного варианта в 1.73 раза.

Соль добавил только ключу 2714. Взял 16 значений по числу shuffle-партиций. Каждому просмотру назначил одну соль, а строку каталога скопировал на все значения, поэтому совпадения и повторы не теряются. Каталог вырос всего на 15 строк — до 10 215.

В первом замере самая большая reduce-задача читала 21.10 млн строк без соли и 5.73 млн с солью. Максимальное время задачи снизилось с 50.9 до 35.4 с. Перекос уменьшился, хотя равномерного распределения по партициям не получилось.

У salting есть своя цена: расчёт соли и дополнительный shuffle. Здесь его объём даже вырос с 360.3 до 388.7 МиБ, но задачи стали лучше распределены. Broadcast для такого каталога оказался проще и быстрее.

Все три результата полностью совпали: 30 070 692 строки с теми же повторами. Замеры и spill — в `results/task2/measurements.csv`, планы — в `results/task2/plans/`.